[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Math/Information_Theory/Information_Theory.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Information Theory

One framework that explains compression limits, communication limits, and — through KL divergence — why cross-entropy is *the* machine learning loss. Four sessions from 'what is a bit?' to the channel coding theorem, with every quantity computed live.

## 0. Introduction

Shannon's move: measure information as **surprise**. Rare events carry more information than expected ones, and $-\log_2 p$ is the unique surprise measure (up to base) that is continuous, decreasing in $p$, and additive over independent events.

## 1. Pre-requisites

[Random Variables](../Analysis/Random_Variables.ipynb) (expectation, distributions); [Independence](../Analysis/Independence.ipynb) for the coding arguments.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def H(p):
    """Entropy in bits of a probability vector."""
    p = np.asarray(p, float); p = p[p > 0]
    return -(p * np.log2(p)).sum()

---
### 🕐 Session 1 of 4 — *Entropy* (~35 min)
**Goal:** quantify average surprise; see entropy as the compression limit.
**Builds on:** [Random Variables](../Analysis/Random_Variables.ipynb). &nbsp; **Feeds into:** Session 2 (KL & cross-entropy).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Entropy</b></summary>

**Timing (~35 min).** 10 min surprise as the right primitive · 8 min the question-counting reading · 10 min the zlib demo, honestly framed · 7 min the source coding theorem.

**Board first — motivate $-\log p$ rather than presenting it.** Ask what properties a measure of "surprise" must have: continuous in $p$, decreasing (rarer is more surprising), and **additive over independent events** (learning two independent things surprises you by the sum). That third requirement forces a logarithm, and $-\log_2 p$ is the unique answer up to base. Deriving the formula from requirements beats announcing it.

**The question-counting reading is what makes entropy concrete.** $H(X)$ is the average number of yes/no questions needed to pin down an outcome *if you ask cleverly*. A fair coin needs 1. A coin that is heads 99% of the time needs far less, because "was it heads?" resolves it almost always. Run a quick round of twenty questions on a skewed distribution if the room is engaged — it makes the compression claim intuitive before it is stated.

**Frame the zlib demo honestly — this is the instruction that matters.** The printed conclusion says the compressor "hugs the entropy floor it can never beat," and the numbers only half support that. At $P(1) = 0.5$, zlib achieves 1.001 against a floor of 1.000 — excellent. But at $P(1) = 0.9$ it needs 0.565 against 0.469, **20% above** the floor, and at $P(1) = 0.99$ it needs 0.115 against 0.081, **42% above**. Do not present those as hugging.

**Then explain the gap, because it is instructive rather than embarrassing.** Two causes. First, zlib is a *general-purpose* compressor built for byte-level repetition, not an optimal code for a known Bernoulli source — an arithmetic coder given the true $p$ would come far closer. Second, and more fundamentally, the source coding theorem is an **asymptotic** statement about long blocks; any practical code carries per-block overhead that matters more as the entropy gets smaller. Ask the room which of the three rows has the smallest absolute overhead — they are all about 0.03–0.1 bits/symbol, and it is only the *relative* cost that explodes as $H$ shrinks.

**The honest summary to give.** Entropy is a floor no code can go below — that part is a theorem and the data respects it. Getting *close* to the floor is an engineering problem that general-purpose tools solve imperfectly. Both halves are worth knowing, and conflating them leaves students thinking compression is a solved problem.

**Ask the room.** "The 0.99 case has entropy 0.081 bits/symbol. What does that mean for a megabyte of such data?" About 81 kbit, roughly 10 kB — a 100× compression, and still 42% worse than optimal. Concrete numbers make the floor feel like an engineering budget rather than a curiosity.
</details>

## 2. Entropy

💡 **Intuition.** Entropy $H(X) = E[-\log_2 p(X)]$ is the **average surprise** of a source — equivalently, the number of yes/no questions you need *on average* to pin down an outcome, when you ask cleverly. A fair coin: 1 bit. A loaded coin: less, because you can exploit the bias. That question-count reading *is* the compression story: you cannot losslessly encode a source below $H$ bits/symbol on average (Shannon's source coding theorem), and Huffman codes get within 1 bit of it.

In [2]:
p = np.linspace(0.001, 0.999, 500)
plt.figure(figsize=(7, 2.6))
plt.plot(p, [H([q, 1-q]) for q in p])
plt.xlabel("P(heads)"); plt.ylabel("H [bits]")
plt.title("Binary entropy: maximal at fair (1 bit), zero when certain")
plt.grid(True); plt.tight_layout(); plt.show()

/tmp/ipykernel_2018428/3520344432.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True); plt.tight_layout(); plt.show()


**What just happened.** The binary entropy curve: **1 bit** at $p = 0.5$, falling to **zero** at both extremes, and symmetric about the middle.

Read the two endpoints first, because they anchor the whole quantity. At $p = 0$ or $p = 1$ the outcome is *certain*, so observing it tells you nothing and the entropy is exactly zero. At $p = 0.5$ you are maximally uncertain and one yes/no question is exactly what it takes. **Entropy measures uncertainty, and uncertainty is maximised by the uniform distribution** — a fact that generalises to any alphabet, and the reason the maximum-entropy principle picks uniform when you know nothing.

**The symmetry is worth a remark too.** $H(0.9) = H(0.1)$: a coin that is 90% heads is exactly as uncertain as one that is 90% tails. Entropy does not care *which* outcome is likely, only how concentrated the distribution is.

**And note how flat the peak is.** Entropy stays above 0.9 bits across roughly $p \in [0.3, 0.7]$ — so a substantially biased coin is still nearly as unpredictable as a fair one. The payoff from exploiting bias only becomes large once the bias is extreme, which is exactly what the next cell measures: at $p = 0.9$ there are still 0.469 bits per symbol to encode, and only at $p = 0.99$ does the floor drop to 0.081.

**Why $-\log_2 p$ and not some other measure of surprise?** Because three requirements force it: surprise should be continuous in $p$, decreasing (rarer events are more surprising), and **additive over independent events** — learning two independent facts should surprise you by the sum of their surprises. Additivity over independence is what demands a logarithm, and the base merely chooses the unit. Base 2 gives bits; base $e$ gives nats. Entropy is then just the *average* surprise, $E[-\log_2 p(X)]$.

In [3]:
# Entropy = compression limit, demonstrated with a real compressor (zlib)
import zlib
for p1 in [0.5, 0.9, 0.99]:
    bits = (rng.random(200_000) < p1).astype(np.uint8)
    packed = np.packbits(bits).tobytes()
    comp = zlib.compress(packed, 9)
    rate = 8 * len(comp) / len(bits)
    print(f"P(1)={p1:4}: entropy {H([p1, 1-p1]):.3f} bits/sym   zlib achieved {rate:.3f} bits/sym")
print("→ a general-purpose compressor hugs the entropy floor it can never beat")

P(1)= 0.5: entropy 1.000 bits/sym   zlib achieved 1.001 bits/sym
P(1)= 0.9: entropy 0.469 bits/sym   zlib achieved 0.565 bits/sym
P(1)=0.99: entropy 0.081 bits/sym   zlib achieved 0.115 bits/sym
→ a general-purpose compressor hugs the entropy floor it can never beat


**What just happened.** A real compressor measured against a theoretical floor — and the result is more interesting than the printed conclusion suggests:

| $P(1)$ | entropy | zlib | above the floor |
|---|---|---|---|
| 0.5 | 1.000 | 1.001 | +0% |
| 0.9 | 0.469 | 0.565 | **+20%** |
| 0.99 | 0.081 | 0.115 | **+42%** |

**The floor holds — that part is a theorem.** No row goes below its entropy, and none ever could: Shannon's source coding theorem says lossless encoding of an i.i.d. source requires at least $H$ bits per symbol on average, and violating it would mean two different inputs mapping to the same output. The 1.001 at $p = 0.5$ is the honest picture of a compressor facing incompressible data — it cannot help and adds a whisker of overhead.

**But "hugs the floor" overstates the other two rows.** 20% and 42% above optimal is a real gap, and it is worth understanding rather than glossing.

Two causes, and they are different in kind. First, **zlib is the wrong tool**: it is a general-purpose compressor built around byte-level repetition (LZ77 plus Huffman), not an optimal code for a known Bernoulli source. An arithmetic coder handed the true $p$ would come far closer to the floor. Second, and more fundamental, the source coding theorem is an **asymptotic** statement about long blocks. Every practical code carries per-symbol and per-block overhead, and notice that the *absolute* overhead is roughly constant across all three rows at 0.03–0.1 bits/symbol — it is only the *relative* cost that explodes as the entropy shrinks toward zero.

**So separate the two claims, because they are frequently conflated.** "You cannot beat entropy" is a theorem, exact and universal. "Real compressors approach entropy" is an engineering statement that is true for well-matched codecs and long inputs, and visibly false for a mismatched general-purpose tool on a highly skewed source. Students who merge them come away believing compression is solved.

**And the practical reading is still striking.** At $P(1) = 0.99$ the data compresses roughly 70× (0.115 bits per symbol against 8 bits stored raw), which is why skewed data is worth compressing at all — even a 42% inefficiency leaves an enormous win. The entropy tells you the prize; the codec decides how much of it you collect.

---
### 🕐 Session 2 of 4 — *KL Divergence & Cross-Entropy* (~40 min)
**Goal:** measure the cost of believing the wrong distribution; derive the ML loss.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (mutual information).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: KL Divergence & Cross-Entropy</b></summary>

**Timing (~40 min).** 10 min the wrong-codebook framing · 10 min Gibbs' inequality · 10 min the demo · 10 min the ML payoff.

**Board first — the framing that makes KL concrete.** The world emits symbols from $p$; you built your code for $q$. You now pay $E_p[-\log_2 q(X)]$ bits per symbol — the **cross-entropy** — instead of the optimal $H(p)$. The overpayment is $D(p\|q)$. Say it as *the price of wrong beliefs, measured in bits*. That reading makes non-negativity obvious before it is proved (you cannot do better than the optimal code) and makes the asymmetry unsurprising (coding $p$-data with a $q$-code is a different mistake from the reverse).

**Do Gibbs' inequality; it is three lines and uses Jensen.** Concavity of $\log$ gives $-D(p\|q) = \sum p\log(q/p) \le \log\sum q = 0$, with equality iff $p = q$. Worth doing because it is the cleanest application of Jensen in the curriculum and because students should see that non-negativity is a *theorem*, not a definition.

**Then the two warnings, and insist on them.** KL is **not symmetric** and violates the triangle inequality, so it is not a distance. The asymmetry has real consequences in practice: minimising $D(p\|q)$ over $q$ (forward KL, what maximum likelihood does) makes $q$ cover all of $p$'s mass and produces blurry over-broad fits; minimising $D(q\|p)$ (reverse KL, what variational inference does) makes $q$ concentrate on one mode. If [Variational Inference](../../Intro_Mach_Learn/Variational_Inference_Flows.ipynb) is on the syllabus, this is the sentence that prepares it.

**The demo's decomposition is the thing to point at.** Every row reads cross-entropy = $H(p)$ + KL, with $H(p) = 1.157$ fixed throughout. So the *only* part a model can improve is the KL term — the floor is set by the data's own entropy and no amount of training removes it. Ask the room what a cross-entropy of 1.157 would mean for a classifier: a perfect model, not a bad one. Students routinely read a nonzero loss as failure, and this decomposition is the cleanest correction available.

**Then the payoff sentence.** Training a classifier with cross-entropy loss is *literally* optimising its codebook for the data distribution. Minimising cross-entropy is minimising KL, since $H(p)$ is constant in the parameters — so every classifier you have ever trained was doing information theory. Ask where the bits went: a model with cross-entropy 2.841 against an entropy of 1.157 is wasting 1.684 bits per symbol on wrong beliefs.

**Connect it forward if time allows.** Fisher information from [Estimation Theory](../Estimation_Theory/Estimation_Theory.ipynb) is the local curvature of KL around $p$ — the two workshops are describing the same object at different scales, one globally and one infinitesimally.
</details>

## 3. Relative Entropy

💡 **Intuition.** Suppose the world emits symbols from $p$ but you built your code (or your model) for $q$. You pay $E_p[-\log_2 q(X)]$ bits per symbol — the **cross-entropy** — instead of the optimal $H(p)$. The overpayment is the KL divergence:
$D(p\|q) = \sum_x p(x) \log \frac{p(x)}{q(x)} \ge 0$ — *the price of wrong beliefs, in bits*. Minimizing cross-entropy in ML is minimizing that price: training a classifier literally optimizes its codebook for the data distribution.

### Proof: $D(p\|q) \ge 0$ (Gibbs' inequality)

Since $\log$ is concave, Jensen's inequality gives
$$-D(p\|q) = \sum_x p(x) \log\frac{q(x)}{p(x)} \le \log \sum_x p(x)\frac{q(x)}{p(x)} = \log \sum_x q(x) = \log 1 = 0,$$
with equality iff $p = q$. $\blacksquare$ Two warnings: $D$ is **not symmetric** and violates the triangle inequality — a directed cost, not a distance.

In [4]:
# Cross-entropy loss IS log-loss with a KL floor
p_true = np.array([0.7, 0.2, 0.1])
qs = {"perfect  q=p": p_true, "close": np.array([0.6, 0.3, 0.1]), "wrong": np.array([0.1, 0.2, 0.7])}
for name, q in qs.items():
    ce = -(p_true * np.log2(q)).sum()
    print(f"{name:14s} cross-entropy {ce:.3f} bits = H(p) {H(p_true):.3f} + KL {ce - H(p_true):.3f}")

perfect  q=p   cross-entropy 1.157 bits = H(p) 1.157 + KL 0.000
close          cross-entropy 1.195 bits = H(p) 1.157 + KL 0.039
wrong          cross-entropy 2.841 bits = H(p) 1.157 + KL 1.684


**What just happened.** Three candidate models scored against the same truth, and every row decomposes the same way:

$$\text{cross-entropy} = \underbrace{H(p)}_{\text{1.157, fixed}} + \underbrace{D(p\|q)}_{\text{0.000 / 0.039 / 1.684}}$$

**The fixed term is the important one.** $H(p) = 1.157$ bits appears in every row and cannot be reduced by any model, because it is the data's *own* uncertainty. Even a perfect model — $q = p$ exactly — pays 1.157 bits per symbol. The only part a model can improve is the KL term.

That has a direct consequence students routinely get wrong: **a nonzero cross-entropy loss is not evidence of a bad model.** A classifier converging to a loss of 1.157 on this data has achieved perfection. If your training loss plateaus above zero, the first question is what $H(p)$ is for your labels — irreducible label noise sets a floor exactly as the noise floor did in [Intro_RNN](../../Intro_Time_Series/Intro_RNN.ipynb)'s forecasting bake-off. Chasing zero loss on noisy labels means overfitting.

**And the KL column is a literal price list.** The "close" model wastes 0.039 bits per symbol; the "wrong" one wastes 1.684 — more than the entire entropy of the source. Multiply by symbols per second and you have the cost of a bad model in bits, which is what KL is: **the price of wrong beliefs**. If you built a code for $q$ and the world emits $p$, that is your overpayment.

**Which is why cross-entropy is *the* machine learning loss, not merely a convenient one.** Since $H(p)$ does not depend on the model parameters, minimising cross-entropy is *exactly* minimising $D(p\|q)$. Training a classifier is optimising its codebook for the data distribution — every classifier you have trained was doing information theory, whether or not it was framed that way.

**Two properties of KL worth carrying, both visible here.** It is non-negative (Gibbs' inequality, via Jensen on the concavity of $\log$), with equality precisely when $q = p$ — which the first row confirms at 0.000. And it is **not symmetric**: $D(p\|q) \neq D(q\|p)$ in general, and it violates the triangle inequality, so it is a *directed cost* rather than a distance.

That asymmetry has real modelling consequences. Minimising $D(p\|q)$ over $q$ — forward KL, what maximum likelihood does — forces $q$ to cover all of $p$'s mass, producing broad, blurry fits. Minimising $D(q\|p)$ — reverse KL, what variational inference does — lets $q$ concentrate on a single mode and ignore the rest. Same two distributions, opposite failure modes, decided entirely by which argument you put first.

---
### 🕐 Session 3 of 4 — *Mutual Information* (~35 min)
**Goal:** quantify what one variable tells you about another; data processing can only lose it.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (coding at a glance).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Mutual Information</b></summary>

**Timing (~35 min).** 8 min the uncertainty-reduction reading · 8 min the two equivalent definitions · 12 min the demo, including its bias · 7 min the data processing inequality.

**Board first — read the definition aloud as a question.** $I(X;Y) = H(X) - H(X|Y)$: how many bits of uncertainty about $X$ does observing $Y$ remove? That framing makes the units meaningful — mutual information is measured in bits, and "2 bits of dependence" means observing $Y$ cuts the space of plausible $X$ values by a factor of four.

**Then the second definition, which is the more powerful one.** $I(X;Y) = D(p_{XY}\|p_Xp_Y)$ — the KL cost of *pretending they are independent*. So mutual information is zero exactly when independence holds, and Session 2's Gibbs inequality immediately gives $I \ge 0$. Students who see this connection get the whole session for free from the previous one.

**Set up the demo as a challenge to correlation.** Have the room predict the three rows before running. The $y = x^2$ case is the one that matters: correlation comes back at $-0.001$, essentially zero, while mutual information reports 0.979 bits. Ask what correlation actually measures — *linear* association — and why a symmetric parabola defeats it: positive and negative $x$ contribute opposite products that cancel exactly. The dependence is total (knowing $x$ determines $y$ up to noise) and the linear measure sees none of it.

**Do not let the "independent" row's 0.003 pass as zero.** It is not: it is the estimator's **bias**. Histogram-based MI is biased *upward* by roughly $(b_x-1)(b_y-1)/(2N\ln 2)$, which with 24×24 bins and $N = 10^5$ is 0.0038 bits — matching the measured 0.003 almost exactly. So the correct reading is "consistent with zero after accounting for a known bias," not "small but nonzero dependence."

**That is the practical warning worth the most in this session.** MI estimated from histograms is *always* biased upward, and the bias grows with the number of bins and shrinks with sample size. So a naive MI estimate will report dependence between genuinely independent variables — and with too many bins, a lot of it. Ask what happens with 100 bins instead of 24: the bias rises to about 0.07 bits, twenty times larger. Anyone using MI for feature selection needs to know this, and most do not.

**Close on the data processing inequality**, which the notebook states without proof. If $X \to Y \to Z$ then $I(X;Z) \le I(X;Y)$: no processing can *create* information about $X$. Every layer of a network, every filter, every feature extractor can only preserve or destroy. Say plainly that this makes "garbage in, garbage out" a theorem rather than a slogan — and note the subtlety that processing can still make information more *accessible* (that is what representation learning does) without ever increasing the total.
</details>

## 4. Mutual Information

💡 **Intuition.** $I(X;Y) = H(X) - H(X|Y)$: how many bits of uncertainty about $X$ does observing $Y$ remove? Equivalently $I(X;Y) = D(p_{XY} \| p_X p_Y)$ — the KL cost of pretending they're independent. Zero iff independent ([Independence](../Analysis/Independence.ipynb), quantified!), and — unlike correlation — it detects *nonlinear* dependence too.

In [5]:
def mi_hist(x, y, bins=24):
    pxy, _, _ = np.histogram2d(x, y, bins=bins, density=False)
    pxy = pxy / pxy.sum()
    px, py = pxy.sum(1, keepdims=True), pxy.sum(0, keepdims=True)
    mask = pxy > 0
    return (pxy[mask] * np.log2(pxy[mask] / (px @ py)[mask])).sum()

N = 100_000
x = rng.standard_normal(N)
pairs = {"independent": rng.standard_normal(N),
         "linear y=x+n": x + 0.5 * rng.standard_normal(N),
         "NONLINEAR y=x²+n": x**2 + 0.5 * rng.standard_normal(N)}
for name, y in pairs.items():
    r = np.corrcoef(x, y)[0, 1]
    print(f"{name:18s} correlation {r:+.3f}   mutual information {mi_hist(x, y):.3f} bits")
print("→ correlation misses y=x²; mutual information doesn't")

independent        correlation +0.000   mutual information 0.003 bits
linear y=x+n       correlation +0.894   mutual information 1.106 bits
NONLINEAR y=x²+n   correlation -0.001   mutual information 0.979 bits
→ correlation misses y=x²; mutual information doesn't


**What just happened.** Three pairs of variables, and the third row is the one that matters:

| relationship | correlation | mutual information |
|---|---|---|
| independent | +0.000 | 0.003 bits |
| linear, $y = x + n$ | +0.894 | 1.106 bits |
| **nonlinear, $y = x^2 + n$** | **−0.001** | **0.979 bits** |

**Correlation reports nothing for $y = x^2$, and it is completely wrong to conclude independence.** Knowing $x$ determines $y$ up to noise — the dependence is essentially total — yet the linear measure sees none of it. The reason is mechanical: correlation averages products $xy$, and for a symmetric parabola the positive and negative $x$ contribute equal and opposite products that cancel exactly. Mutual information reports 0.979 bits, nearly as much dependence as the linear case.

This is worth stating as a warning rather than a curiosity: **"uncorrelated" does not mean "independent."** A correlation matrix showing zeros can hide arbitrarily strong nonlinear structure, and a feature discarded for low correlation may be the most informative one you had. Mutual information is zero **iff** independence holds — that is the definition $I(X;Y) = D(p_{XY}\|p_Xp_Y)$, the KL cost of pretending independence, combined with Session 2's Gibbs inequality.

**Now read the "independent" row correctly, because 0.003 is not zero and that is not an accident.** Histogram-based MI estimators are biased **upward**, by roughly
$$\frac{(b_x-1)(b_y-1)}{2N\ln 2} = \frac{23 \times 23}{2 \times 10^5 \times 0.693} = 0.0038 \text{ bits}$$
with the 24×24 bins and $N = 10^5$ used here — which matches the measured 0.003 almost exactly. So the honest reading is "consistent with zero once the known estimator bias is accounted for," not "a small amount of dependence."

**That bias is the practical trap in this session.** It grows with the number of bins and shrinks with sample size, so a naive MI estimate will *always* report some dependence between genuinely independent variables. Use 100 bins instead of 24 and the bias rises to about 0.07 bits — twenty times larger, and easily mistaken for a real signal. Anyone using MI for feature selection needs a null baseline: shuffle one variable and estimate MI again to measure your own estimator's floor, exactly as you would compare a detector against chance.

**And the structural result the next cell states.** The **data processing inequality**: if $X \to Y \to Z$ then $I(X;Z) \le I(X;Y)$. No processing can create information about $X$ — every network layer, filter, and feature extractor can only preserve or destroy it. That makes "garbage in, garbage out" a theorem. The subtlety worth adding is that processing *can* make information more **accessible** without increasing it, which is precisely what representation learning does: same bits, better arranged.

**Data processing inequality** (stated): if $X \to Y \to Z$ is a Markov chain (Z computed from Y alone), then $I(X;Z) \le I(X;Y)$ — **no processing can create information about $X$**. Deep networks, filters, features: every stage can only preserve or destroy. This is the information-theoretic backbone of representation learning, and the reason 'garbage in, garbage out' is a theorem.

---
### 🕐 Session 4 of 4 — *Coding at a Glance* (~35 min)
**Goal:** the two Shannon theorems, one channel capacity computed, and the SNR connection.
**Builds on:** Sessions 1–3.

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: Coding at a Glance</b></summary>

**Timing (~35 min).** 5 min restating source coding · 12 min the channel coding theorem and why it shocked people · 10 min the BSC capacity curve · 8 min the Gaussian formula and link budgets.

**Lead with why the channel coding theorem was shocking, because the shock is the content.** Before Shannon, the engineering consensus was that a noisy channel imposed a floor on error rate: push harder and you get fewer errors, but never zero. Shannon proved that is wrong. **Noise does not limit reliability — it limits *rate*.** Below capacity you can achieve *arbitrarily small* error probability; above it, nothing works. Let the room sit with how counterintuitive that is before explaining the mechanism.

**Then the mechanism, which is a law-of-large-numbers argument.** Use long codewords. Over a long block the noise concentrates ([Independence](../Analysis/Independence.ipynb)'s LLN), so received sequences cluster into nearly-disjoint balls around each codeword, and decoding is "which ball am I in?" Capacity counts how many disjoint balls fit into the output space. That picture makes both halves of the theorem intuitive — below capacity the balls fit and separate, above it they must overlap.

**Read the BSC curve at three points, and make the room predict the third.** At $\varepsilon = 0$, capacity is 1 bit per use — a clean channel. At $\varepsilon = 0.5$, capacity is **zero**: the output is independent of the input, and no coding recovers anything. Then the good question: what is the capacity at $\varepsilon = 1$? It is **1 bit**, not zero — a channel that flips *every* bit is perfectly reliable, you simply invert the output. Students almost always say zero, and the correction lands the point that capacity measures *information transfer*, not fidelity.

**Note where the curve's shape comes from.** $C = 1 - H(\varepsilon)$ is Session 1's binary entropy, subtracted. The channel costs you exactly the entropy of its own noise, which is a satisfying closure of the workshop's four sessions into one formula.

**Then the Gaussian formula, framed as the currency of engineering.** $C = \frac{1}{2}\log_2(1+\mathrm{SNR})$ bits per use. Ask what doubling the SNR buys: half a bit — *logarithmic* returns on power. Contrast with the linear return on bandwidth or antennas, which is exactly the asymmetry that motivated [MIMO](../../Intro_DSP/MIMO_Communications.ipynb). Every link budget in [Digital Communications](../../Intro_DSP/Digital_Communications.ipynb) is this formula with units attached.

**Be honest about what the theorem does not give you.** It is an *existence* proof: it says good codes exist, and says almost nothing about how to build or decode them. Shannon's argument uses random codes, which are optimal and computationally hopeless to decode. Closing the gap took fifty years — [turbo, LDPC, and polar codes](../../Intro_DSP/Channel_Coding.ipynb) — and that half-century between knowing a thing is possible and knowing how to do it is worth naming explicitly.
</details>

## 5. The Two Theorems

**Source coding** (S1's floor): lossless compression needs $\ge H$ bits/symbol.

**Channel coding.** A noisy channel has capacity $C = \max_{p_X} I(X;Y)$; *any* rate below $C$ is achievable with vanishing error (via long random-ish codes), and no rate above it is. The shocking part: noise does **not** cap reliability — only *rate*.

💡 **Intuition.** Long codewords let the law of large numbers ([Independence](../Analysis/Independence.ipynb)) concentrate the noise: typical received sequences cluster into disjoint balls around codewords, and decoding is 'which ball am I in?'. Capacity counts how many disjoint balls fit.

In [6]:
# Capacity of the binary symmetric channel: C = 1 − H(ε)
eps = np.linspace(0, 0.5, 200)
C = 1 - np.array([H([e, 1-e]) if 0 < e else 0 for e in eps])
plt.figure(figsize=(7, 2.6))
plt.plot(eps, C)
plt.xlabel("bit-flip probability ε"); plt.ylabel("capacity [bits/use]")
plt.title("BSC capacity: 1 bit when clean, 0 at ε = ½ (pure noise)")
plt.grid(True); plt.tight_layout(); plt.show()

/tmp/ipykernel_2018428/1934163155.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True); plt.tight_layout(); plt.show()


**The formula on every comms slide.** For the Gaussian channel with signal-to-noise ratio SNR: $C = \tfrac12 \log_2(1 + \mathrm{SNR})$ bits per use — bandwidth and SNR are the currency of every link budget in [Digital Communications](../../Intro_DSP/Digital_Communications.ipynb).

## 6. Conclusion

Entropy = surprise = compression floor; KL = the bits you waste believing $q$ when truth is $p$ (cross-entropy loss, demystified); mutual information = dependence in bits, immune to nonlinearity, only ever destroyed by processing; capacity = the rate ceiling noise imposes. Four numbers that govern every pipeline in this curriculum.

---
## Where next

- [Digital Communications](../../Intro_DSP/Digital_Communications.ipynb) — engineering toward capacity.
- [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb) — cross-entropy at work, at scale.
- [Estimation Theory](../Estimation_Theory/Estimation_Theory.ipynb) — Fisher information: KL's local curvature.